 This notebook sets up a sophisticated Retrieval-Augmented Generation (RAG) system using CrewAI, which orchestrates multiple AI agents to collaboratively answer questions.

 # Router Agent:
 Decides whether to use the vectorstore or web search.
# Retriever Agent:
Fetches information using either the PDFSearchTool or SerperDevTool based on the Router's decision.
# Grader Agent:
Assesses the relevance of the retrieved information to the user's question.
# Hallucination Grader:
Verifies the factual accuracy and grounding of the retrieved content.
# Final Answer Agent:
Synthesizes all information to formulate the final, comprehensive answer.

In [ ]:
pip install crewai crewai_tools

  Using cached openai-1.83.0-py3-none-any.whl.metadata (25 kB)
  Using cached opentelemetry_api-1.34.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_http-1.34.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.34.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached opentelemetry_exporter_otlp_proto_common-1.34.1-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_proto-1.34.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_semantic_conventions-0.55b1-py3-none-any.whl.metadata (2.5 kB)
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.39.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.38.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.3

In [ ]:
import os
#import gradio as gr
import requests
from pydantic import BaseModel, Field
from crewai_tools import PDFSearchTool
from crewai_tools import SerperDevTool
from crewai import Crew, Task, Agent, LLM
from crewai.tools import tool

In [ ]:
from google.colab import userdata
#os.environ["OPENAI_MODEL_NAME"] = "gpt-4-turbo"
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')
os.environ["SERPER_API_KEY"]=userdata.get('SERPER_API_KEY')

In [ ]:
from crewai_tools import SerperDevTool

tool = SerperDevTool(
    search_url="https://google.serper.dev/scholar",
    n_results=2,
)

print(tool.run(search_query="ChatGPT"))

{'searchParameters': {'q': 'ChatGPT', 'type': 'search', 'num': 2, 'engine': 'google'}, 'knowledgeGraph': {'title': 'ChatGPT', 'type': 'Software', 'website': '', 'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRCppx-iYdqsyRCPqK6lNLjExFyYPmkYtA&s=0', 'description': 'ChatGPT is a generative artificial intelligence chatbot developed by OpenAI. It was released in November 2022. It uses generative pre-trained transformers, such as GPT-5, to generate text, speech, and images in response to user prompts.', 'descriptionSource': 'Wikipedia', 'descriptionLink': 'https://en.wikipedia.org/wiki/ChatGPT', 'attributes': {'Programming language': 'Python', 'Developer': 'OpenAI and Sam Altman', 'Initial release date': 'November 30, 2022'}}, 'organic': [{'title': 'ChatGPT', 'link': 'https://chatgpt.com/', 'snippet': 'ChatGPT is your AI chatbot for everyday use. Chat with the most advanced AI to explore ideas, solve problems, and learn faster.', 'position': 1, 'sitelinks': [{'title': 'Lo

In [ ]:
# Download PDF if not already present
def download_pdf():
    pdf_url = 'https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf'
    if not os.path.exists('attention_is_all_you_need.pdf'):
        response = requests.get(pdf_url)
        with open('attention_is_all_you_need.pdf', 'wb') as file:
            file.write(response.content)

In [ ]:
llm = LLM(
    model="openai/gpt-4o-mini",
    temperature=0.1,        # Higher for more creative outputs
    timeout=120,           # Seconds to wait for response
    max_tokens=1000,       # Maximum length of response
    seed=42               # For reproducible results
)

In [ ]:
# Tools Configuration
# RAG Tool
rag_tool = PDFSearchTool(
        pdf='/content/attention-is-all-you-need-Paper.pdf',
        config=dict(
            llm=dict(
                provider="openai",
                config=dict(
                    model="gpt-4-turbo",
                ),
            ),
            embedder=dict(
                provider="huggingface",
                config=dict(
                    model="BAAI/bge-small-en-v1.5",
                ),
            ),
        )
    )

     # Web Search Tool
web_search_tool = SerperDevTool()




In [ ]:
# Router Tool
from crewai.tools import tool as crew_tool # Re-import and alias the tool decorator
class RouterToolInput(BaseModel):
    question: str = Field(description="The user's question to be routed.")

@crew_tool("Router Tool") # Use the aliased decorator
def router_tool(question: str):
    """Router Function: Routes user question to a vectorstore or web search based on keywords.
    The input should be the user's question as a string."""
    keywords = ['self-attention', 'transformer', 'attention', 'language model']
    if any(keyword in question.lower() for keyword in keywords):
        return 'vectorstore'
    return 'web_search'

In [ ]:
# Agent Definitions
Router_Agent = Agent(
        role='Router',
        goal='Route user question to a vectorstore or web search',
        backstory=(
            "You are an expert at routing a user question to a vectorstore or web search. "
            "Use the vectorstore for questions related to Retrieval-Augmented Generation. "
            "Be flexible in interpreting keywords related to these topics."
        ),
        verbose=True,
        allow_delegation=False,
        llm=llm,
    )

Retriever_Agent = Agent(
        role="Retriever",
        goal="Use retrieved information to answer the question",
        backstory=(
            "You are an assistant for question-answering tasks. "
            "Provide clear, concise answers using retrieved context."
        ),
        verbose=True,
        allow_delegation=False,
        llm=llm,
    )

Grader_agent = Agent(
        role='Answer Grader',
        goal='Filter out irrelevant retrievals',
        backstory=(
            "You are a grader assessing the relevance of retrieved documents. "
            "Evaluate if the document contains keywords related to the user question."
        ),
        verbose=True,
        allow_delegation=False,
        llm=llm,
    )

Hallucination_Grader = Agent(
        role="Hallucination Grader",
        goal="Verify answer factuality",
        backstory=(
            "You are responsible for ensuring the answer is grounded in facts "
            "and directly addresses the user's question."
        ),
        verbose=True,
        allow_delegation=False,
        llm=llm,
    )

Final_Answer_Agent = Agent(
        role="Final Answer Agent",
        goal="Provide a comprehensive and accurate response",
        backstory=(
            "You synthesize information from various sources to create "
            "a clear, concise, and informative answer to the user's question."
        ),
        verbose=True,
        allow_delegation=False,
        llm=llm,
    )



In [ ]:
# Task Definitions
router_task = Task(
        description=(
            "Analyze the keywords in the question {question}. "
            "Decide whether it requires a vectorstore search or web search."
        ),
        expected_output="Return 'websearch' or 'vectorstore'",
        agent=Router_Agent,
        tools=[router_tool],
    )

retriever_task = Task(
        description=(
            "Retrieve information for the question {question} "
            "using either web search or vectorstore based on router task."
        ),
        expected_output="Provide retrieved information",
        agent=Retriever_Agent,
        context=[router_task],
        tools=[rag_tool, web_search_tool],
    )

grader_task = Task(
        description="Evaluate the relevance of retrieved content for the question {question}",
        expected_output="Return 'yes' or 'no' for relevance",
        agent=Grader_agent,
        context=[retriever_task],
    )

hallucination_task = Task(
        description="Verify if the retrieved answer is factually grounded",
        expected_output="Return 'yes' or 'no' for factual accuracy",
        agent=Hallucination_Grader,
        context=[grader_task],
    )

answer_task = Task(
        description=(
            "Generate a final answer based on retrieved and verified information. "
            "Perform additional search if needed."
        ),
        expected_output="Provide a clear, concise answer",
        agent=Final_Answer_Agent,
        context=[hallucination_task],
        tools=[web_search_tool],
    )


In [ ]:
# Main RAG Function
def run_rag_pipeline(question):
    # Download PDF if not exists
    download_pdf()

    # Setup tools
    rag_tool, web_search_tool = setup_tools()
    tools = (rag_tool, web_search_tool)

    # Create agents
    agents = create_agents()

    # Create tasks
    tasks = create_tasks(agents, tools)

    # Create Crew
    rag_crew = Crew(
        agents=agents,
        tasks=tasks,
        verbose=True,
    )

    # Run the pipeline
    try:
        result = rag_crew.kickoff(inputs={"question": "What is deeplearning?"})
        return result
    except Exception as e:
        return f"An error occurred: {str(e)}"

In [ ]:
# Create Crew
rag_crew = Crew(
        agents=[Router_Agent, Retriever_Agent, Grader_agent, Hallucination_Grader, Final_Answer_Agent],
        tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task],
        verbose=True,
    )

In [ ]:
result = rag_crew.kickoff(inputs={"question": "What is Neural network?"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  e421304a-3ccc-47ce-9b89-862c206c23e9                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
    "name": "Crew Created",
    "context": {
        "trace_id": "0xecb2b88bf2e9a3ce78109c089e418216",
        "span_id": "0x1a71bce00a3dc08d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-02-07T04:35:56.500726Z",
    "end_time": "2026-02-07T04:35:56.505197Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "crewai_version": "1.9.3",
        "python_version": "3.12.12",
        "crew_key": "373278eeefd701a34f62882e032ef41d",
        "crew_id": "e421304a-3ccc-47ce-9b89-862c206c23e9",
        "crew_fingerprint": "bebabfbb-7deb-4bc6-a40e-f14cfb63de99",
        "crew_process": "sequential",
        "crew_memory": false,
        "crew_number_of_tasks": 5,
        "crew_number_of_agents": 5,
        "crew_fingerprint_created_at": "2026-02-07T04:35:52.701674",
        "crew_agents": "[{\"key\": \"9774ae80be11ee92873f52c2a052c3d1\", \"id\": \"5518d74a-32ae-447b-a41b-04b26620d4f2\", \"role\": 

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the keywords in the question What is Neural network?. Decide whether it requires a vectorstore   │
│  search or web search.                                                                                          │
│  ID: b36345ce-e244-4339-8e4d-d0c98cd5f905                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router                                                                                                  │
│                                                                                                                 │
│  Task: Analyze the keywords in the question What is Neural network?. Decide whether it requires a vectorstore   │
│  search or web search.                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool router_tool executed with result: web_search...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: router_tool                                                                                              │
│  Args: {'question': 'What is Neural network?'}                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (105 outputs left out)

{
    "name": "Task Execution",
    "context": {
        "trace_id": "0x95c018c4e7dd58f0bdb7c9b0d2a0162a",
        "span_id": "0x6ad0a965df15b93c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-02-07T04:36:16.709919Z",
    "end_time": "2026-02-07T04:36:19.078866Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "crew_key": "373278eeefd701a34f62882e032ef41d",
        "crew_id": "e421304a-3ccc-47ce-9b89-862c206c23e9",
        "crew_fingerprint": "bebabfbb-7deb-4bc6-a40e-f14cfb63de99",
        "task_key": "5427b44568c27171253ff3c9c4f4c95e",
        "task_id": "55b3c3ae-0171-49ee-be38-583cafeccfb1",
        "agent_fingerprint": "27eb2dc1-6202-42f1-991f-db1a230dfee3",
        "agent_role": "Final Answer Agent",
        "task_fingerprint": "501b4241-6b07-4cb2-bab6-d2d8b500fa34"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "service.name": "cre

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Generate a final answer based on retrieved and verified information. Perform additional search if needed.      │
│  Agent:                                                                                                         │
│  Final Answer Agent                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 